The goal of this notebook is to perform a baseline RAG to detect which condition a patient has in the NTDS notes set. An LLM will be given a long note and will need to perform RAG on the specific note itself to help determine this. It should only retrieve from the specific note used.

In [1]:
import numpy as np
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
import json

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:


notes = pd.read_csv("data/synthetic_ntds_trauma_notes_gemini.csv", index_col = "encounter_id")

In [ ]:
with open("./data/ntds_18_complications.json", "r") as f:
    complications_info = json.load(f)
complications_info

In [5]:
def format_complications_info(complications_info):
    output_lines = []
    
    for comp in complications_info:
        # Add label and short_definition
        output_lines.append(f"{comp['label']}: {comp['short_definition']}")
        
        output_lines.append("Positive Note Clues:")
        for i, clue in enumerate(comp['positive_note_clues'], 1):
            output_lines.append(f"{i}. {clue}")
        
        # Add blank line between complications
        output_lines.append("")
    
    return "\n".join(output_lines)

# Test the function
complications_str = format_complications_info(complications_info)
print(complications_str)

Acute Kidney Injury: New kidney dysfunction during this hospitalization, such as a rise in serum creatinine or oliguria/anuria, not clearly present before admission.
Positive Note Clues:
1. developed acute kidney injury with rising creatinine
2. oliguria requiring nephrology consultation
3. initiated dialysis for new renal failure

Alcohol Withdrawal Syndrome: Clinical alcohol withdrawal that began after admission, with symptoms such as tremor, agitation, hallucinations, or withdrawal seizures.
Positive Note Clues:
1. placed on alcohol withdrawal protocol with high CIWA scores
2. developed agitation and tremors consistent with alcohol withdrawal
3. treated with benzodiazepines for withdrawal symptoms

Acute Respiratory Distress Syndrome: Acute hypoxemic respiratory failure with bilateral lung infiltrates not fully explained by cardiac failure or fluid overload, meeting ARDS criteria during the stay.
Positive Note Clues:
1. worsening hypoxemia with bilateral infiltrates consistent with 

In [6]:
# I can feed in the actual possible conditions as a system prompt. Then I can feed in the
# notes as a user prompt 
test_template = ChatPromptTemplate(
    [
    ("system", """You are an expert registrar who is highly experienced at meeting the
    National Trauma Data Standard (NTDS). You will be provided a patient's medical note
    from UCSD Health, a level 1 health center. Please determine which of the conditions the 
    patient has. Below are the possible complications:""" + '\n' + complications_str),
    ("human", "{note}"),
    ("system", """The following chunks were to be relevant to the final outputs.
     Please output your answers as a JSON with a 1 indicating the patient has the disorder
     and a zero indicating that the patient does not. Include all 18 fields.""")
    ]
)

In [7]:
sample_note = notes.loc[0, 'note_text']
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, 
                                          chunk_overlap = 200, 
                                          separators = ["\n\n", "\n", " ", ""])
chunked = splitter.split_text(sample_note)
chunked

['**ED TRAUMA H&P:**\nThis 79-year-old male was brought to the trauma bay via EMS after a significant fall from a height of approximately 15 feet while working on his roof. Per EMS report, he was found alert but confused at the scene by family and complained of diffuse body pain. On arrival, initial vital signs were heart rate 97 bpm, blood pressure 120/78 mmHg (MAP 92 mmHg), respiratory rate 22 breaths/min, and oxygen saturation 95% on a 4L nasal cannula, later titrated to 6L to maintain saturation >94%.',
 'Primary survey was completed rapidly per ATLS protocol. Airway was patent and protected. Breath sounds were clear bilaterally, though shallow, with no obvious respiratory distress. Cardiovascularly, peripheral pulses were palpable and strong, skin was warm and dry, capillary refill brisk. Neurologically, he presented with a GCS of 14 (E4V4M6), oriented to person but confused to place and time, without obvious focal deficits upon initial assessment. Gross deformities were noted to 

In [8]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
import os

# Initialize ChromaDB with persistence
PERSIST_DIRECTORY = "./ntds_embeddings"

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    collection_name="ntds_notes",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIRECTORY
)

# Initialize text splitter with same parameters as before
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200, 
    separators=["\n\n", "\n", " ", ""]
)

# Process all notes
all_chunks = []
all_metadatas = []

print(f"Processing {len(notes)} notes...")
for idx, (encounter_id, row) in enumerate(notes.iterrows()):
    note_text = row['note_text']
    
    # Chunk the note
    chunks = splitter.split_text(note_text)
    
    # Create metadata for each chunk
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({
            'encounter_id': str(encounter_id),
            'chunk_index': chunk_idx,
            'total_chunks': len(chunks)
        })
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(notes)} notes...")

print(f"\nAdding {len(all_chunks)} chunks to ChromaDB...")
vectorstore.add_texts(texts=all_chunks, metadatas=all_metadatas)

print(f"Successfully created ChromaDB with {len(all_chunks)} chunks")
print(f"Persisted to: {PERSIST_DIRECTORY}")

Processing 50 notes...
Processed 10/50 notes...
Processed 20/50 notes...
Processed 30/50 notes...
Processed 40/50 notes...
Processed 50/50 notes...

Adding 141 chunks to ChromaDB...
Successfully created ChromaDB with 141 chunks
Persisted to: ./ntds_embeddings


## RAG with Metadata Filtering

Now we'll implement single-note RAG using ChromaDB's metadata filtering. This ensures we only retrieve chunks from the specific patient note being analyzed.

In [9]:
# Reload the existing vectorstore (no need to recreate)
vectorstore = Chroma(
    collection_name="ntds_notes",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIRECTORY
)

print(f"Loaded vectorstore with collection: ntds_notes")
print(f"Total documents in collection: {vectorstore._collection.count()}")

Loaded vectorstore with collection: ntds_notes
Total documents in collection: 282


In [10]:
# Test metadata filtering - retrieve only from encounter_id 0
test_encounter_id = "0"
test_query = "kidney injury or renal failure"

print(f"Testing retrieval for encounter {test_encounter_id}")
print(f"Query: '{test_query}'")
print("="*60)

results = vectorstore.similarity_search(
    test_query,
    k=5,
    filter={"encounter_id": test_encounter_id}
)

# Display results to verify filtering works
for i, doc in enumerate(results):
    print(f"\nChunk {i+1}:")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content[:200]}...")
    print()

Testing retrieval for encounter 0
Query: 'kidney injury or renal failure'

Chunk 1:
Metadata: {'encounter_id': '0', 'total_chunks': 15, 'chunk_index': 2}
Content: CT imaging from head to pelvis demonstrated multiple injuries consistent with severe blunt trauma, including comminuted bilateral rib fractures (R: 5-9, L: 6-8), a nondisplaced pelvic rami fracture on...


Chunk 2:
Metadata: {'chunk_index': 2, 'encounter_id': '0', 'total_chunks': 15}
Content: CT imaging from head to pelvis demonstrated multiple injuries consistent with severe blunt trauma, including comminuted bilateral rib fractures (R: 5-9, L: 6-8), a nondisplaced pelvic rami fracture on...


Chunk 3:
Metadata: {'total_chunks': 15, 'chunk_index': 7, 'encounter_id': '0'}
Content: The patient's overall status showed gradual improvement on ICU Day 2. Hemodynamically, he became more stable, no longer requiring fluid boluses, and his blood pressure maintained within target range. ...


Chunk 4:
Metadata: {'chunk_index': 7, 'enco

In [11]:
def retrieve_relevant_chunks(vectorstore, encounter_id, query, k=5):
    results = vectorstore.similarity_search(
        query,
        k=k,
        filter={"encounter_id": str(encounter_id)}
    )
    return [doc.page_content for doc in results]

print("Helper function 'retrieve_relevant_chunks' created successfully!")

Helper function 'retrieve_relevant_chunks' created successfully!


In [14]:
# Test retrieval for different complication types
test_queries = {
    "aki": "acute kidney injury rising creatinine dialysis renal failure",
    "delirium": "confusion disoriented CAM-ICU delirium",
    "dvt": "deep vein thrombosis DVT anticoagulation ultrasound",
    "pe": "pulmonary embolism PE CT angiogram"
}

encounter_id = "0"
print(f"Testing complication-specific retrieval for encounter {encounter_id}\n")

for comp_id, query in test_queries.items():
    print(f"\n{'='*60}")
    print(f"Testing {comp_id.upper()} query")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    chunks = retrieve_relevant_chunks(vectorstore, encounter_id, query, k=1)
    for i, chunk in enumerate(chunks, 1):
        print(f"\nChunk {i}: {chunk[:300]}...")
        if i < len(chunks):
            print(f"{'~'*60}")

Testing complication-specific retrieval for encounter 0


Testing AKI query
Query: 'acute kidney injury rising creatinine dialysis renal failure'

Chunk 1: toileting with incentive spirometry was initiated. Invasive monitoring with an arterial line was placed for continuous blood pressure surveillance. Strict input/output monitoring was maintained via Foley catheter. Initial laboratory trends showed a stable hemoglobin and creatinine. Prophylactic low ...

Testing DELIRIUM query
Query: 'confusion disoriented CAM-ICU delirium'

Chunk 1: At the time of discharge, this 79-year-old male demonstrated an excellent recovery from his significant fall from height and associated injuries, which included multiple bilateral rib fractures, a nondisplaced left pelvic rami fracture, a T10 compression fracture, and a small subarachnoid hemorrhage...

Testing DVT query
Query: 'deep vein thrombosis DVT anticoagulation ultrasound'

Chunk 1: Primary survey was completed rapidly per ATLS protocol. Airway w

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize LLM (using Google Gemini - API key already loaded from .env)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# Create a comprehensive query that includes all complication keywords
complication_query = " ".join([
    comp['label'] + " " + " ".join(comp['positive_note_clues'])
    for comp in complications_info
])

print(f"Complication query length: {len(complication_query)} characters")
print(f"Sample of query: {complication_query[:200]}...")

# Retrieve relevant chunks for a specific encounter
encounter_id = "0"
relevant_chunks = retrieve_relevant_chunks(
    vectorstore,
    encounter_id,
    complication_query,
    k=1 
)

print(f"\nRetrieved {len(relevant_chunks)} chunks for encounter {encounter_id}")
print(f"\nFirst chunk preview:\n{relevant_chunks[0][:300]}...")

In [ ]:
# Format retrieved chunks for the LLM
retrieved_context = "\n\n---\n\n".join([
    f"Relevant Section {i+1}:\n{chunk}"
    for i, chunk in enumerate(relevant_chunks)
])

# Create the RAG prompt template
rag_template = ChatPromptTemplate(
    [
        ("system", """You are an expert registrar who is highly experienced at meeting the
        National Trauma Data Standard (NTDS). You will be provided with relevant sections
        from a patient's medical note from UCSD Health, a level 1 health center. Please
        determine which of the conditions the patient has. Below are the possible complications:""" + '\n' + complications_str),
        ("human", "{retrieved_chunks}"),
        ("system", """The above chunks were determined to be relevant to identifying complications.
        Please output your answers as a JSON with a 1 indicating the patient has the disorder
        and a 0 indicating that the patient does not. Include all 18 fields.

        Output format: {"aki": 0, "aws": 0, "ards": 0, "cardiac_arrest_cpr": 0, "cauti": 0, "delirium": 0, "dvt": 0, "mi": 0, "osteomyelitis": 0, "pressure_ulcer": 0, "pe": 0, "severe_sepsis": 0, "stroke_cva": 0, "superficial_ssi": 0, "unplanned_icu_admission": 0, "unplanned_intubation": 0, "unplanned_or_visit": 0, "vap": 0}""")
    ]
)

# Generate response
print(f"Sending {len(retrieved_context)} characters of context to LLM...")
response = llm.invoke(rag_template.format_messages(retrieved_chunks=retrieved_context))
print("\nLLM Response:")
print(response.content)